# GPCP monthly precipitation — data exploration
Source file: `data/gpcp_monthly_precipitation.csv`, extracted by `data/collect_data.py` from NOAA PSL GPCP v2.3 (10 reference points, one per country).
Goal: understand the file, check its quality and see what the rainfall looks like. Nothing is dropped or changed in the data here.

In [1]:
# Core libraries for cleaning and exploring data
import pandas as pd
import numpy as np
import pyarrow
# Read the CSV file into a DataFrame.
df_raw = pd.read_csv("../data/gpcp_monthly_precipitation.csv")

### loading the data
- The file is a **CSV**, so `pd.read_csv` is the right reader.
- It loads **3,180 rows x 7 columns** in *long format*: one row = one country + one month. That is 10 countries x 318 months.

In [2]:
#copy the data frame to a new variable to avoid changing the original data frame
df = df_raw.copy()
df.head()

,date,country,original_lat,original_lon,grid_lat,grid_lon,precipitation_mm_per_day
0,2000-01-01,Egypt,26.8206,30.8025,26.25,31.25,0.143663
1,2000-02-01,Egypt,26.8206,30.8025,26.25,31.25,0.023449
2,2000-03-01,Egypt,26.8206,30.8025,26.25,31.25,0.010577
3,2000-04-01,Egypt,26.8206,30.8025,26.25,31.25,0.003601
4,2000-05-01,Egypt,26.8206,30.8025,26.25,31.25,0.012697


### first look at the rows
- The first rows are **Egypt, from 2000-01**. Rainfall is tiny (`0.14`, `0.02`, `0.01` mm/day), which already shows how dry this point is.
- The point requested for Egypt (`26.82, 30.80`) is not the point used: GPCP returned the **nearest grid cell** (`26.25, 31.25`).

In [3]:
print("Shape:", df.shape)
df.tail(3)

Shape: (3180, 7)


,date,country,original_lat,original_lon,grid_lat,grid_lon,precipitation_mm_per_day
3177,2026-04-01,Vietnam,14.0583,108.2772,13.75,108.75,1.301758
3178,2026-05-01,Vietnam,14.0583,108.2772,13.75,108.75,4.840898
3179,2026-06-01,Vietnam,14.0583,108.2772,13.75,108.75,2.398930


### size and ordering of the file
- Shape confirmed: **(3180, 7)**.
- The file is sorted by country and then by date: the last rows are **Vietnam, 2026-04 to 2026-06** (`1.30`, `4.84` and `2.40` mm/day).

In [4]:
# Column names
print(f"Column names: {df.columns.tolist()}")

Column names: ['date', 'country', 'original_lat', 'original_lon', 'grid_lat', 'grid_lon', 'precipitation_mm_per_day']


### column names
| Column | Meaning |
|---|---|
| `date` | Month (first day of the month) |
| `country` | Label of the reference point (`USA_Midwest`, `South_Africa`, `Vietnam`... not the FAOSTAT names) |
| `original_lat`, `original_lon` | The point that was requested (country centroid) |
| `grid_lat`, `grid_lon` | The GPCP grid cell that was actually used (nearest, 2.5° grid) |
| `precipitation_mm_per_day` | Monthly mean rainfall rate, in mm per day |

In [5]:
df_raw.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 3180 entries, 0 to 3179
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      3180 non-null   str    
 1   country                   3180 non-null   str    
 2   original_lat              3180 non-null   float64
 3   original_lon              3180 non-null   float64
 4   grid_lat                  3180 non-null   float64
 5   grid_lon                  3180 non-null   float64
 6   precipitation_mm_per_day  3180 non-null   float64
dtypes: float64(5), str(2)
memory usage: 227.8 KB


### data types and memory
- 5 `float64` columns and 2 text columns (`date`, `country`).
- All 7 columns have **3,180 non-null** values.
- Memory is about **228 KB**.

In [6]:
# Null counts and duplicates
print("Total nulls   :", df_raw.isna().sum().sum())
print("Duplicate rows:", df_raw.duplicated().sum())
print("Duplicate date + country pairs:", df_raw.duplicated(subset=["date", "country"]).sum())

# Exact memory per column, in KB
(df_raw.memory_usage(deep=True) / 1024).round(1)

Total nulls   : 0
Duplicate rows: 0
Duplicate date + country pairs: 0


Index                        0.1
date                        55.9
country                     47.5
original_lat                24.8
original_lon                24.8
grid_lat                    24.8
grid_lon                    24.8
precipitation_mm_per_day    24.8
dtype: float64

### missing values, duplicates and memory per column
- **0 missing values, no duplicate rows and no duplicate `date` + `country` pairs.**
- The text columns cost the most memory: `date` (`55.9` KB) and `country` (`47.5` KB), because the same strings repeat on every row.

In [7]:
# 1. date: text -> datetime, 2. country: text -> category, 3. float64 -> float32 (the source file is float32)
df["date"] = pd.to_datetime(df["date"])
df["country"] = df["country"].astype("category")
for col in ["original_lat", "original_lon", "grid_lat", "grid_lon", "precipitation_mm_per_day"]:
    df[col] = df[col].astype("float32")

print(df.dtypes)
print("Before:", round(df_raw.memory_usage(deep=True).sum() / 1024, 1), "KB")
print("After :", round(df.memory_usage(deep=True).sum() / 1024, 1), "KB")

# Did float32 change the rainfall values?
print("Max |float32 - float64| rainfall:", (df["precipitation_mm_per_day"].astype("float64") - df_raw["precipitation_mm_per_day"]).abs().max())
print("Max |float32 - float64| original_lat:", (df["original_lat"].astype("float64") - df_raw["original_lat"]).abs().max())

date                        datetime64[us]
country                           category
original_lat                       float32
original_lon                       float32
grid_lat                           float32
grid_lon                           float32
precipitation_mm_per_day           float32
dtype: object
Before: 227.8 KB
After : 90.3 KB
Max |float32 - float64| rainfall: 9.252929693559508e-07
Max |float32 - float64| original_lat: 1.3488769496916575e-06


### fixing the data types
- `date` is now `datetime64`, `country` is `category` (10 labels) and the five numeric columns are `float32`.
- Memory went from **227.8 KB to 90.3 KB (about 60% less)**.
- `float32` changes the rainfall by less than `0.000001` mm/day, so the conversion is safe.

In [8]:
# Coverage per country: rows, first and last month
cov = df.groupby("country", observed=True)["date"].agg(rows="size", first="min", last="max")
print(cov)

expected = pd.date_range(df["date"].min(), df["date"].max(), freq="MS")
print("Months expected per country:", len(expected))
print("Countries with a gap       :", [c for c, g in df.groupby("country", observed=True) if len(expected.difference(g["date"])) > 0])
print("All dates on day 1         :", (df["date"].dt.day == 1).all())

              rows      first       last
country                                 
Australia      318 2000-01-01 2026-06-01
Brazil         318 2000-01-01 2026-06-01
Egypt          318 2000-01-01 2026-06-01
India          318 2000-01-01 2026-06-01
Indonesia      318 2000-01-01 2026-06-01
Kenya          318 2000-01-01 2026-06-01
Peru           318 2000-01-01 2026-06-01
South_Africa   318 2000-01-01 2026-06-01
USA_Midwest    318 2000-01-01 2026-06-01
Vietnam        318 2000-01-01 2026-06-01
Months expected per country: 318
Countries with a gap       : []
All dates on day 1         : True


### is every country complete?
- Every country has **318 rows, from 2000-01 to 2026-06**, with **no gaps** and all dates on day 1.
- The panel is perfectly balanced: 10 x 318 = 3,180 rows.

In [9]:
# Reference points: the point we asked for vs the grid cell GPCP returned
coords = ["original_lat", "original_lon", "grid_lat", "grid_lon"]
print("Max distinct coordinates per country (should be 1):", df_raw.groupby("country")[coords].nunique().max().max())

p = df_raw.groupby("country")[coords].first()
p["lon_as_0_360"] = (p["original_lon"] % 360).round(2)
p["dlat"] = (p["grid_lat"] - p["original_lat"]).round(2)
p["dlon"] = (p["grid_lon"] - p["lon_as_0_360"]).round(2)
p.round(2)

Max distinct coordinates per country (should be 1): 1


,original_lat,original_lon,grid_lat,grid_lon,lon_as_0_360,dlat,dlon
country,,,,,,,
Australia,-25.27,133.78,-26.25,133.75,133.78,-0.98,-0.03
Brazil,-14.24,-51.93,-13.75,308.75,308.07,0.48,0.68
Egypt,26.82,30.80,26.25,31.25,30.80,-0.57,0.45
India,20.59,78.96,21.25,78.75,78.96,0.66,-0.21
Indonesia,-0.79,113.92,-1.25,113.75,113.92,-0.46,-0.17
Kenya,-1.29,36.82,-1.25,36.25,36.82,0.04,-0.57
Peru,-9.19,-75.02,-8.75,283.75,284.98,0.44,-1.23
South_Africa,-30.56,22.94,-31.25,23.75,22.94,-0.69,0.81
USA_Midwest,41.88,-87.63,41.25,271.25,272.37,-0.63,-1.12


### requested point against grid cell
- Each country has **exactly one** coordinate pair, so the file really is one fixed point per country.
- The grid cell is never more than **`0.98`° in latitude and `1.23`° in longitude** away from the requested point, which is within half a cell (a cell is 2.5° wide).
- Example: Peru is the furthest in longitude (`-1.23`°) and Australia in latitude (`-0.98`°).

In [10]:
# Longitude convention: the grid uses 0-360, the original points use -180..180
print("grid_lon range    :", df["grid_lon"].min(), "->", df["grid_lon"].max())
print("original_lon range:", df["original_lon"].min(), "->", df["original_lon"].max())
print("Countries with grid_lon > 180:", df.loc[df["grid_lon"] > 180, "country"].unique().tolist())
print("Largest grid offset (degrees):", p[["dlat", "dlon"]].abs().max().tolist())

# GPCP is a 2.5 degree grid with cell centres at 1.25, 3.75, 6.25 ... (and the same pattern for longitude)
print("grid_lat on cell centres:", ((df["grid_lat"] - 1.25) % 2.5 == 0).all())
print("grid_lon on cell centres:", ((df["grid_lon"] - 1.25) % 2.5 == 0).all())

grid_lon range    : 23.75 -> 308.75
original_lon range: -87.6298 -> 133.7751
Countries with grid_lon > 180: ['Brazil', 'Peru', 'USA_Midwest']
Largest grid offset (degrees): [0.98, 1.23]
grid_lat on cell centres: True
grid_lon on cell centres: True


### longitude convention
- `grid_lon` goes up to `308.75`, while `original_lon` is between `-87.63` and `133.78`: the grid uses **0 to 360** and the requested points use **-180 to 180**.
- **Brazil, Peru and USA_Midwest** have `grid_lon` above 180 (for example Brazil `308.75` against `-51.93`).
- Before mapping the points or joining on coordinates, one convention must be chosen. Otherwise these three countries would be drawn in the wrong place.
- All grid coordinates fall exactly on 2.5° cell centres, so the nearest-cell lookup behaved as expected.

In [11]:
# Summary stats for the rainfall column
df["precipitation_mm_per_day"].describe().round(3)

count    3180.000
mean        3.552
std         4.108
min         0.000
25%         0.292
50%         1.887
75%         5.674
max        25.414
Name: precipitation_mm_per_day, dtype: float64

### rainfall summary
- Rainfall ranges from `0.0` to **`25.41` mm/day**, with a mean of `3.55`.
- The **median (`1.89`) is far below the mean**, and the 75th percentile is `5.67`: the distribution is **strongly right-skewed**, with a few very wet months and many dry ones.

In [12]:
# Rainfall per country: mean, spread, extremes and months that are almost dry
g = df.groupby("country", observed=True)["precipitation_mm_per_day"]
summary = pd.DataFrame({
    "mean": g.mean(), "std": g.std(), "min": g.min(), "max": g.max(),
    "months < 0.1": g.apply(lambda s: (s < 0.1).sum()),
    "zero months": g.apply(lambda s: (s == 0).sum()),
}).round(2).sort_values("mean", ascending=False)
summary

,mean,std,min,max,months < 0.1,zero months
country,,,,,,
Indonesia,10.22,3.81,1.03,18.59,0,0
Vietnam,5.61,4.55,0.02,25.41,3,0
Peru,5.13,2.77,0.95,14.34,0,0
Brazil,4.30,4.11,0.00,18.23,57,0
India,3.23,4.56,0.00,20.23,78,3
USA_Midwest,2.98,1.42,0.54,8.29,0,0
Kenya,2.41,1.92,0.26,12.78,0,0
South_Africa,1.05,0.97,0.01,4.93,26,0
Australia,0.58,0.86,0.00,6.33,120,3


### rainfall by country
| Country | Mean (mm/day) | Min | Max | Months < 0.1 |
|---|---|---|---|---|
| Indonesia | 10.22 | 1.03 | 18.59 | 0 |
| Vietnam | 5.61 | 0.02 | 25.41 | 3 |
| Peru | 5.13 | 0.95 | 14.34 | 0 |
| Brazil | 4.30 | 0.00 | 18.23 | 57 |
| India | 3.23 | 0.00 | 20.23 | 78 |
| USA_Midwest | 2.98 | 0.54 | 8.29 | 0 |
| Kenya | 2.41 | 0.26 | 12.78 | 0 |
| South_Africa | 1.05 | 0.01 | 4.93 | 26 |
| Australia | 0.58 | 0.00 | 6.33 | 120 |
| Egypt | 0.03 | 0.00 | 0.37 | 307 |

- **Egypt is almost always dry**: 307 of 318 months are below `0.1` mm/day and the maximum is `0.37`. This point carries almost no rainfall signal.
- **Australia** is also very dry (120 months below `0.1`), and India and Brazil have long dry seasons.
- Only India and Australia have months with **exactly zero** rainfall (3 each).

In [13]:
# Seasonal pattern: mean mm/day for each calendar month
season = (df.assign(month=df["date"].dt.month)
            .pivot_table(index="month", columns="country", values="precipitation_mm_per_day", aggfunc="mean", observed=True)
            .round(2))
print(season)

# Wettest and driest calendar month of every country
print(pd.DataFrame({"wettest month": season.idxmax(), "driest month": season.idxmin()}))

country  Australia  Brazil  Egypt  India  Indonesia  Kenya  Peru  \
month                                                              
1             0.64    9.72   0.05   0.31      11.43   1.97  7.16   
2             1.24    8.56   0.03   0.32      12.45   1.93  8.53   
3             0.69    7.19   0.05   0.53      13.48   3.29  8.30   
4             0.38    3.59   0.01   0.47      13.93   5.39  5.30   
5             0.41    0.63   0.02   0.56      10.69   2.60  3.32   
6             0.42    0.18   0.01   5.95       7.75   1.24  2.92   
7             0.27    0.06   0.01  12.58       5.54   1.15  2.05   
8             0.17    0.18   0.01   9.72       5.58   1.35  1.71   
9             0.26    1.17   0.01   7.17       6.92   1.38  2.77   
10            0.58    4.01   0.02   1.24       9.70   2.10  5.02   
11            1.01    7.29   0.05   0.25      12.74   3.60  6.51   
12            0.90    8.92   0.04   0.05      12.10   2.80  7.72   

country  South_Africa  USA_Midwest  Vietnam  
m

### seasonal pattern
- The seasonal cycle is **much larger than anything else** in this file, and it is different in every country:
  - **India:** monsoon from June to September, wettest in July (`12.58`), almost dry in December (`0.05`).
  - **Vietnam:** wettest in September and October (`11.25`, `11.16`), driest in February (`0.55`).
  - **Brazil:** wettest in January (`9.72`), driest in July (`0.06`).
  - **Kenya:** two rainy seasons, a main peak in April (`5.39`) and a second in November (`3.60`).
  - **USA_Midwest:** the flattest cycle (`2.26` to `3.98`), with a June peak.
  - **Indonesia:** wet all year (never below `5.5` mm/day on average in any month).
- **Implication:** raw rainfall mostly shows the seasons. To see an ENSO effect, each month must be compared with the normal for **its own calendar month** (an anomaly).

In [14]:
# Extreme months: the 5 wettest country-months in the file
df.nlargest(5, "precipitation_mm_per_day")[["date", "country", "precipitation_mm_per_day"]]

,date,country,precipitation_mm_per_day
2992,2010-11-01,Vietnam,25.413780
1224,2022-07-01,India,20.232925
2978,2009-09-01,Vietnam,19.623520
3111,2020-10-01,Vietnam,18.754725
755,2009-12-01,Indonesia,18.587933


### extreme months
- The five wettest country-months are **Vietnam 2010-11 (`25.41`)**, India 2022-07 (`20.23`), Vietnam 2009-09 (`19.62`), Vietnam 2020-10 (`18.75`) and Indonesia 2009-12 (`18.59`).
- Vietnam holds **three of the top five**, which fits its strong autumn peak.

In [15]:
# Cross-check with the other files: same end date as the events file, and what comes after it
ev = pd.read_csv("../data/el-nino-southern-oscillation-enso-el-nino-and-la-nina-events.csv", parse_dates=["date"])
nino = pd.read_excel("../data/ENSO_NINO_data.xlsx")
nino_end = pd.Timestamp(year=int(nino["YR"].iloc[-1]), month=int(nino["MON"].iloc[-1]), day=1)
print("GPCP ends  :", df["date"].max().date())
print("ONI file ends  :", ev["date"].max().date())
print("Nino file ends :", nino_end.date())
print("Months of GPCP covered by the ONI file:", df["date"].drop_duplicates().isin(ev["date"]).sum(), "of", df["date"].nunique())

GPCP ends  : 2026-06-01
ONI file ends  : 2026-06-01
Nino file ends : 2026-10-01
Months of GPCP covered by the ONI file: 318 of 318


### cross-check with the other files
- GPCP ends in **2026-06**, the same month as the events file, and **all 318 GPCP months are covered** by that file.
- The Niño file runs **four months longer** (to 2026-10), so rainfall is the limiting series for the 2026 event: there is no GPCP data yet for the months in which the Niño anomalies are largest.

In [16]:
# Save an untouched Parquet copy of the raw file (checkpoint before any cleaning)
import os
os.makedirs("../data/parquet", exist_ok=True)
df_raw.to_parquet("../data/parquet/gpcp_raw.parquet")

back = pd.read_parquet("../data/parquet/gpcp_raw.parquet")
print("Round trip identical:", back.equals(df_raw))
print("CSV    :", round(os.path.getsize("../data/gpcp_monthly_precipitation.csv") / 1024, 1), "KB")
print("Parquet:", round(os.path.getsize("../data/parquet/gpcp_raw.parquet") / 1024, 1), "KB")

Round trip identical: True
CSV    : 183.4 KB
Parquet: 37.9 KB


### Parquet checkpoint
- The raw file is saved as `data/parquet/gpcp_raw.parquet`, and reading it back returns an **identical** DataFrame.
- Size drops from **183.4 KB to 37.9 KB**. This is the untouched copy; cleaning is done later in SQL.

### Conclusion: coverage and data quality

| Check | Result |
|---|---|
| Rows x columns | 3,180 x 7 |
| Panel | 10 reference points x 318 months, balanced |
| Period | 2000-01 to 2026-06, monthly |
| Missing values / duplicates / gaps | none / none / none |
| Unit | mm per day (monthly mean rate) |

### Conclusion: decisions for the cleaning step

| Topic | Action |
|---|---|
| `date`, `country` | Convert to `datetime` and `category` (done here) |
| Longitudes | Use one convention (0-360 or -180..180) before mapping or joining on coordinates |
| Country names | Align with FAOSTAT names before joining (`Vietnam` / `Viet Nam`, `USA_Midwest` / `United States of America`, ...) as the README notes |
| Rainfall values | Compute anomalies against the monthly normal of each country; raw values mainly show the seasons |
| Egypt | Almost no rainfall (307 of 318 months below 0.1 mm/day): not usable for an ENSO rainfall signal; Australia is also very dry |
| Period | Ends in 2026-06 (the ONI file ends the same month; the Niño file runs to 2026-10) |